# Conditional GAN (pix2pix generator + PatchGAN discriminator)

Part of **FL-YOLO-DADP** (Chen et al., *Improved Federated Learning with Differential Privacy for YOLO Detector on Phase Images*, CDSR 2025). This is one of the four generative baselines in Table 1.

The generator learns to map a 3-channel bounding-box mask (one channel per class: RBC, WBC, PLT) to a phase image. It is trained on real phase images with YOLO labels (`generator_train`), then used to generate a synthetic version of every frame of the three agents' train/valid splits.

**Cells:** *Train* → *Synthesize* (visual check on one frame) → *Generate* (writes `../data/synthetic_raw/GANs/agent_<k>_synthesized/`). Afterwards, from the repository root:

```bash
python data_prep/build_synthetic_dataset.py --raw data/synthetic_raw/GANs --agents data/agents --out data/synthetic/GANs
python baselines/train_yolo.py --data data/synthetic/GANs/data.yaml --epochs 20 --name synthetic_GANs
```

Paths are relative to this folder (`generative_models/`):

| path | content |
|---|---|
| `../data/generator_train/{train,valid,test}` | real frames + YOLO labels used to train the generator |
| `../data/agents/agent_<k>` | per-agent shards (input of *Generate*) |
| `../data/synthetic_raw/GANs` | output of *Generate* |
| `../runs/generative_models/GANs` | generator checkpoints |


In [ ]:
import os
os.makedirs("../runs/generative_models/GANs/models/20epochs", exist_ok=True)


### Train

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.optim as optim
import cv2
from PIL import Image
from tqdm import tqdm
import random

# Set random seed for reproducibility
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ----------------------------------
# 1. Dataset
# ----------------------------------

class YOLOConditionalDataset(Dataset):
    """
    Loads images & YOLO-format bounding-box annotations,
    creates a multi-channel mask (one channel per class).
    Returns (mask, real_image).
    """

    def __init__(self, images_dir, labels_dir, image_size=(512, 384), num_classes=3, transforms=None):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_size = image_size  # Corrected to (width, height)
        self.num_classes = num_classes
        self.transforms = transforms
        
        # Collect all image paths
        self.image_paths = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))
        
        # Default transform: ToTensor
        if self.transforms is None:
            self.transforms = T.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        filename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.labels_dir, filename + ".txt")

        # Load the real image (RGB)
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        # Create bounding-box mask
        mask = np.zeros((h, w, self.num_classes), dtype=np.uint8)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            for line in lines:
                cls_id, x_c, y_c, bw, bh = line.strip().split()
                cls_id = int(cls_id)
                x_c, y_c, bw, bh = float(x_c), float(y_c), float(bw), float(bh)

                # Convert normalized coords to pixel coords
                x_center = int(x_c * w)
                y_center = int(y_c * h)
                box_w = int(bw * w)
                box_h = int(bh * h)

                x1 = max(0, x_center - box_w // 2)
                y1 = max(0, y_center - box_h // 2)
                x2 = min(w, x_center + box_w // 2)
                y2 = min(h, y_center + box_h // 2)

                mask[y1:y2, x1:x2, cls_id] = 1

        # Resize image and mask to (512, 384)
        image = image.resize(self.image_size, Image.BILINEAR)  # Resize to (width, height)
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)  # Resize to (width, height)

        # Convert to Tensor
        image = self.transforms(image)  # Apply transforms (e.g., ToTensor)
        mask = torch.from_numpy(np.transpose(mask, (2, 0, 1))).float()  # [num_classes, H, W], 0/1

        return mask, image

# ----------------------------------
# 2. Model: Generator (U-Net style)
# ----------------------------------

class DownBlock(nn.Module):
    """A single down block (Conv -> BN -> LeakyReLU) with stride=2."""
    def __init__(self, in_ch, out_ch, apply_norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        ]
        if apply_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    """
    A single up block:
      1) TransposeConv (in -> out)
      2) (option) concat skip
      3) Conv (out + skip_ch -> out)
      4) (option) dropout
    """
    def __init__(self, in_ch, out_ch, skip_ch=0, dropout=False):
        super().__init__()
        self.up_conv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.up_bn = nn.BatchNorm2d(out_ch)
        self.post_conv = nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1)
        self.post_bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(0.5) if dropout else nn.Identity()

    def forward(self, x, skip=None):
        x = self.up_conv(x)
        x = self.up_bn(x)
        x = self.relu(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.post_conv(x)
        x = self.post_bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x

class Pix2PixGenerator(nn.Module):
    """
    A 6-layer pix2pix generator that correctly aligns channel counts
    for (384,512) input. Each up block is explicitly defined with its
    skip input channel. No final skip to the input.
    """
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.d1 = DownBlock(in_channels, 64,  apply_norm=False)
        self.d2 = DownBlock(64, 128)
        self.d3 = DownBlock(128, 256)
        self.d4 = DownBlock(256, 512)
        self.d5 = DownBlock(512, 512)
        self.d6 = DownBlock(512, 512, apply_norm=False)
        self.u1 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        self.u2 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        self.u3 = UpBlock(in_ch=512, out_ch=256, skip_ch=256, dropout=True)
        self.u4 = UpBlock(in_ch=256, out_ch=128, skip_ch=128)
        self.u5 = UpBlock(in_ch=128, out_ch=64, skip_ch=64)
        self.u6 = UpBlock(in_ch=64, out_ch=64, skip_ch=0, dropout=False)
        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, x):
        d1_out = self.d1(x)
        d2_out = self.d2(d1_out)
        d3_out = self.d3(d2_out)
        d4_out = self.d4(d3_out)
        d5_out = self.d5(d4_out)
        d6_out = self.d6(d5_out)
        u1_out = self.u1(d6_out, d5_out)
        u2_out = self.u2(u1_out, d4_out)
        u3_out = self.u3(u2_out, d3_out)
        u4_out = self.u4(u3_out, d2_out)
        u5_out = self.u5(u4_out, d1_out)
        u6_out = self.u6(u5_out, None)
        return self.final(u6_out)

class PatchDiscriminator(nn.Module):
    """
    PatchGAN discriminator from pix2pix:
    Input: concatenation of mask and image -> shape = [N, (in_channels + out_channels), H, W].
    Output: real/fake patch logits, shape ~ [N, 1, H/16, W/16].
    """
    def __init__(self, in_channels=6): 
        super().__init__()
        def disc_block(in_feat, out_feat, norm=True):
            layers = [nn.Conv2d(in_feat, out_feat, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.BatchNorm2d(out_feat))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *disc_block(in_channels, 64, norm=False),
            *disc_block(64, 128),
            *disc_block(128, 256),
            nn.Conv2d(256, 512, 4, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, stride=1, padding=1)  # Output channels = 1
        )

    def forward(self, x):
        return self.model(x)

# ----------------------------------
# 4. Training Loop
# ----------------------------------

def train_cgan():
    # Paths
    train_images_dir = "../data/generator_train/train/images"
    train_labels_dir = "../data/generator_train/train/labels"
    val_images_dir = "../data/generator_train/valid/images"
    val_labels_dir = "../data/generator_train/valid/labels"
    
    # Hyperparameters
    image_size = (512, 384)  # Corrected to (width, height)
    batch_size = 4
    lr = 2e-4
    beta1, beta2 = 0.5, 0.999
    num_epochs = 20
    lambda_L1 = 100.0  # Weight for L1 loss
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Dataset & DataLoader
    transform = T.ToTensor()
    train_dataset = YOLOConditionalDataset(
        images_dir=train_images_dir,
        labels_dir=train_labels_dir,
        image_size=image_size,
        transforms=transform
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    val_dataset = YOLOConditionalDataset(
        images_dir=val_images_dir,
        labels_dir=val_labels_dir,
        image_size=image_size,
        transforms=transform
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Initialize models
    G = Pix2PixGenerator(in_channels=3, out_channels=3).to(device)
    D = PatchDiscriminator(in_channels=3+3).to(device)  # 3 for mask + 3 for image
    
    # Optimizers
    opt_G = optim.Adam(G.parameters(), lr=lr, betas=(beta1, beta2))
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(beta1, beta2))
    
    # Loss functions
    criterion_GAN = nn.BCEWithLogitsLoss()  # For adversarial
    criterion_L1 = nn.L1Loss()             # For reconstruction
    
    # Label tensors
    real_label = 1.
    fake_label = 0.
    
    # Track best validation accuracy
    best_val_acc = -1.0
    save_dir = "../runs/generative_models/GANs/models/20epochs"
    os.makedirs(save_dir, exist_ok=True)
    
    for epoch in range(num_epochs):
        G.train()
        D.train()
        train_loss = 0.0
        train_acc = 0.0
        
        # Training loop
        train_loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] [Train]", leave=True)
        for mask, real_img in train_loop:
            mask, real_img = mask.to(device), real_img.to(device)
            batch_size_ = mask.size(0)
            
            # Train Discriminator
            concat_real = torch.cat((mask, real_img), dim=1)
            real_validity = D(concat_real)
            real_gt = torch.ones_like(real_validity).to(device) * real_label
            d_real_loss = criterion_GAN(real_validity, real_gt)
            
            fake_img = G(mask)
            concat_fake = torch.cat((mask, fake_img.detach()), dim=1)
            fake_validity = D(concat_fake)
            fake_gt = torch.zeros_like(fake_validity).to(device) * fake_label
            d_fake_loss = criterion_GAN(fake_validity, fake_gt)
            d_loss = 0.5 * (d_real_loss + d_fake_loss)
            
            opt_D.zero_grad()
            d_loss.backward()
            opt_D.step()
            
            # Train Generator
            concat_fake_for_G = torch.cat((mask, fake_img), dim=1)
            fake_validity_for_G = D(concat_fake_for_G)
            real_gt_for_G = torch.ones_like(fake_validity_for_G).to(device) * real_label
            g_adv_loss = criterion_GAN(fake_validity_for_G, real_gt_for_G)
            l1_loss = criterion_L1(fake_img, real_img) * lambda_L1
            g_loss = g_adv_loss + l1_loss
            
            opt_G.zero_grad()
            g_loss.backward()
            opt_G.step()
            
            train_loss += d_loss.item() + g_loss.item()
            train_acc += 1 - (d_loss.item() + g_loss.item())  # Placeholder for accuracy
            
            train_loop.set_postfix({
                "D_loss": f"{d_loss.item():.4f}",
                "G_loss": f"{g_loss.item():.4f}"
            })
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = train_acc / len(train_loader)
        
        # Validation loop
        G.eval()
        val_loss = 0.0
        val_acc = 0.0
        
        with torch.no_grad():
            val_loop = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] [Val]", leave=True)
            for mask, real_img in val_loop:
                mask, real_img = mask.to(device), real_img.to(device)
                
                # Generate fake image
                fake_img = G(mask)
                
                # Calculate L1 loss as a placeholder for accuracy
                l1_loss = criterion_L1(fake_img, real_img)
                val_loss += l1_loss.item()
                val_acc += 1 - l1_loss.item()
                
                val_loop.set_postfix({
                    "Val Loss": f"{val_loss / len(val_loader):.4f}",
                    "Val Acc": f"{val_acc / len(val_loader):.4f}"
                })
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_acc / len(val_loader)
        
        # Save last model
        last_G_path = os.path.join(save_dir, f"generator_last.pth")
        last_D_path = os.path.join(save_dir, f"discriminator_last.pth")
        torch.save(G.state_dict(), last_G_path)
        torch.save(D.state_dict(), last_D_path)
        
        # Save best model
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            best_G_path = os.path.join(save_dir, f"generator_best.pth")
            best_D_path = os.path.join(save_dir, f"discriminator_best.pth")
            torch.save(G.state_dict(), best_G_path)
            torch.save(D.state_dict(), best_D_path)
            print(f"Best model saved with Val Acc: {best_val_acc:.4f}")
    
    print(f"Training complete. Last model saved to: {last_G_path}")

if __name__ == "__main__":
    train_cgan()

### Synthesize

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# 1. Define your Pix2PixGenerator
# ------------------------------
class DownBlock(nn.Module):
    """A single down block (Conv -> BN -> LeakyReLU) with stride=2."""
    def __init__(self, in_ch, out_ch, apply_norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        ]
        if apply_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    """
    A single up block:
      1) TransposeConv (in -> out)
      2) (option) concat skip
      3) Conv (out + skip_ch -> out)
      4) (option) dropout
    This ensures channel counts match properly.
    """
    def __init__(self, in_ch, out_ch, skip_ch=0, dropout=False):
        super().__init__()
        self.up_conv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.up_bn = nn.BatchNorm2d(out_ch)
        
        # after we upsample and possibly concat skip, we do a 3x3 conv to get back to out_ch
        self.post_conv = nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1)
        self.post_bn = nn.BatchNorm2d(out_ch)
        
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(0.5) if dropout else nn.Identity()

    def forward(self, x, skip=None):
        """
        x: upsampled input, shape=[B, in_ch, H, W]
        skip: skip-connection features, shape=[B, skip_ch, H, W], or None
        """
        # 1) TransposeConv
        x = self.up_conv(x)
        x = self.up_bn(x)
        x = self.relu(x)
        
        # 2) Optionally concat skip
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        
        # 3) 3x3 conv to reduce channels back to out_ch
        x = self.post_conv(x)
        x = self.post_bn(x)
        x = self.relu(x)
        
        # 4) dropout if needed
        x = self.dropout(x)
        return x

class Pix2PixGenerator(nn.Module):
    """
    A 6-layer pix2pix generator that correctly aligns channel counts
    for (384,512) input. Each up block is explicitly defined with its
    skip input channel. No final skip to the input.
    """
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        
        # ----------------
        # 1) Down blocks
        # ----------------
        # d1 out = 64
        self.d1 = DownBlock(in_channels, 64,  apply_norm=False)
        # d2 out = 128
        self.d2 = DownBlock(64, 128)
        # d3 out = 256
        self.d3 = DownBlock(128, 256)
        # d4 out = 512
        self.d4 = DownBlock(256, 512)
        # d5 out = 512
        self.d5 = DownBlock(512, 512)
        # d6 out = 512 (no skip above this)
        self.d6 = DownBlock(512, 512, apply_norm=False)

        # ----------------
        # 2) Up blocks
        # ----------------
        # up1: in=512, skip=512 -> out=512
        self.u1 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        # up2: in=512, skip=512 -> out=512
        self.u2 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        # up3: in=512, skip=256 -> out=256
        self.u3 = UpBlock(in_ch=512, out_ch=256, skip_ch=256, dropout=True)
        # up4: in=256, skip=128 -> out=128
        self.u4 = UpBlock(in_ch=256, out_ch=128, skip_ch=128)
        # up5: in=128, skip=64 -> out=64
        self.u5 = UpBlock(in_ch=128, out_ch=64, skip_ch=64)
        # up6: in=64, skip=None -> out=64
        self.u6 = UpBlock(in_ch=64, out_ch=64, skip_ch=0, dropout=False)

        # final 1x1 conv
        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, x):
        # 1) Encoder
        d1_out = self.d1(x)    # 64
        d2_out = self.d2(d1_out)   # 128
        d3_out = self.d3(d2_out)   # 256
        d4_out = self.d4(d3_out)   # 512
        d5_out = self.d5(d4_out)   # 512
        d6_out = self.d6(d5_out)   # 512

        # 2) Decoder
        # up1 => skip with d5 => cat(512,512)=1024 => out=512
        u1_out = self.u1(d6_out, d5_out)
        # up2 => skip with d4 => cat(512,512)=1024 => out=512
        u2_out = self.u2(u1_out, d4_out)
        # up3 => skip with d3 => cat(512,256)=768 => out=256
        u3_out = self.u3(u2_out, d3_out)
        # up4 => skip with d2 => cat(256,128)=384 => out=128
        u4_out = self.u4(u3_out, d2_out)
        # up5 => skip with d1 => cat(128,64)=192 => out=64
        u5_out = self.u5(u4_out, d1_out)
        # up6 => no skip => out=64
        u6_out = self.u6(u5_out, None)

        return self.final(u6_out)

class PatchDiscriminator(nn.Module):
    """
    PatchGAN discriminator from pix2pix:
    Input: concatenation of mask and image -> shape = [N, (in_channels + out_channels), H, W].
    Output: real/fake patch logits, shape ~ [N, 1, H/16, W/16].
    """
    def __init__(self, in_channels=6): 
        # By default: mask(3 ch) + image(3 ch) = 6
        super().__init__()
        
        def disc_block(in_feat, out_feat, norm=True):
            layers = [nn.Conv2d(in_feat, out_feat, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.BatchNorm2d(out_feat))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        
        self.model = nn.Sequential(
            *disc_block(in_channels, 64, norm=False),  # (6 -> 64)
            *disc_block(64, 128),
            *disc_block(128, 256),
            # stride=1 for the last layer (scale down less)
            nn.Conv2d(256, 512, 4, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, stride=1, padding=1)  # Output channels = 1
        )

    def forward(self, x):
        return self.model(x)

# ------------------------------
# 2. Instantiate generator & load weights
# ------------------------------
model = Pix2PixGenerator(in_channels=3, out_channels=3).to(device)
checkpoint_path = "../runs/generative_models/GANs/models/20epochs/generator_best.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# ------------------------------
# 3. Prepare test input
#    *IMPORTANT*: Must match the training resolution (256x256)
# ------------------------------
test_image_path = "../data/generator_train/test/images/test-agg1-2067.png"
test_image = cv2.imread(test_image_path)
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)

# Resize to 256x256 (or the resolution used in training)
test_image = cv2.resize(test_image, (512, 384), interpolation=cv2.INTER_AREA)

# Convert to float32, scale to [0,1]
test_image = test_image.astype(np.float32) / 255.0

# [H, W, C] -> [C, H, W] -> add batch dim -> to device
test_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)

# ------------------------------
# 4. Forward pass
# ------------------------------
with torch.no_grad():
    # The generator typically outputs in [-1, 1] range
    out = model(test_tensor)

# Convert from [-1,1] to [0,1] for visualization
out = (out + 1) / 2.0
out = torch.clamp(out, 0, 1)

out_np = out.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H,W,3]
gray = out_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
out_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)
print('out_np.shape', out_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(out_np), np.max(out_np))

identical_across_channels = (out_np[:,:,0] == out_np[:,:,1]).all() \
                            and (out_np[:,:,1] == out_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

plt.imshow(out_np)
# plt.title("Synthesized Image (Generator Output)")
plt.axis("off")
plt.show()

### Generate

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as T

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# 1. Define Pix2PixGenerator (from your GAN code)
# ------------------------------
class DownBlock(nn.Module):
    """A single down block (Conv -> BN -> LeakyReLU) with stride=2."""
    def __init__(self, in_ch, out_ch, apply_norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        ]
        if apply_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    """
    A single up block:
      1) TransposeConv (in -> out)
      2) (option) concat skip
      3) Conv (out + skip_ch -> out)
      4) (option) dropout
    """
    def __init__(self, in_ch, out_ch, skip_ch=0, dropout=False):
        super().__init__()
        self.up_conv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.up_bn = nn.BatchNorm2d(out_ch)
        self.post_conv = nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1)
        self.post_bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(0.5) if dropout else nn.Identity()

    def forward(self, x, skip=None):
        x = self.up_conv(x)
        x = self.up_bn(x)
        x = self.relu(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.post_conv(x)
        x = self.post_bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x

class Pix2PixGenerator(nn.Module):
    """
    A 6-layer pix2pix generator.
    """
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.d1 = DownBlock(in_channels, 64,  apply_norm=False)
        self.d2 = DownBlock(64, 128)
        self.d3 = DownBlock(128, 256)
        self.d4 = DownBlock(256, 512)
        self.d5 = DownBlock(512, 512)
        self.d6 = DownBlock(512, 512, apply_norm=False)
        self.u1 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        self.u2 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        self.u3 = UpBlock(in_ch=512, out_ch=256, skip_ch=256, dropout=True)
        self.u4 = UpBlock(in_ch=256, out_ch=128, skip_ch=128)
        self.u5 = UpBlock(in_ch=128, out_ch=64, skip_ch=64)
        self.u6 = UpBlock(in_ch=64, out_ch=64, skip_ch=0, dropout=False)
        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, x):
        d1_out = self.d1(x)
        d2_out = self.d2(d1_out)
        d3_out = self.d3(d2_out)
        d4_out = self.d4(d3_out)
        d5_out = self.d5(d4_out)
        d6_out = self.d6(d5_out)
        u1_out = self.u1(d6_out, d5_out)
        u2_out = self.u2(u1_out, d4_out)
        u3_out = self.u3(u2_out, d3_out)
        u4_out = self.u4(u3_out, d2_out)
        u5_out = self.u5(u4_out, d1_out)
        u6_out = self.u6(u5_out, None)
        return self.final(u6_out)

# ------------------------------
# 2. Load the Trained GAN Model
# ------------------------------
def load_trained_gan_model(model_path, device):
    model = Pix2PixGenerator(in_channels=3, out_channels=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()  # Set the model to evaluation mode
    return model

# ------------------------------
# 3. Synthesize Images with GAN
# ------------------------------
def synthesize_images_gan(model, source_images_dir, target_images_dir, image_size=(512, 384)):
    # Create target directory if it doesn't exist
    os.makedirs(target_images_dir, exist_ok=True)
    
    # Collect all image paths
    image_paths = sorted([os.path.join(source_images_dir, fname) for fname in os.listdir(source_images_dir) if fname.endswith('.png')])
    
    for img_path in image_paths:
        # Load the image
        test_image = cv2.imread(img_path)
        test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB
        test_image = test_image.astype(np.float32) / 255.0  # Scale to [0, 1]
        
        # Resize the image to the desired size
        test_image = cv2.resize(test_image, image_size, interpolation=cv2.INTER_LINEAR)
        
        # Convert to tensor: [H, W, C] --> [C, H, W], then add batch dimension
        mask_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)
        
        # Generate synthetic image
        with torch.no_grad():
            fake_image = model(mask_tensor)  # Shape: [1, 3, H, W]
        
        # Convert from [-1,1] to [0,1] for visualization
        fake_image = (fake_image + 1) / 2.0
        fake_image = torch.clamp(fake_image, 0, 1)
        
        # Convert to NumPy for saving
        fake_image_np = fake_image.squeeze(0).cpu().numpy()  # Shape: [3, H, W]
        fake_image_np = np.transpose(fake_image_np, (1, 2, 0))  # Shape: [H, W, 3]
        gray = np.mean(fake_image_np, axis=2, keepdims=True)    # shape [H,W,1]
        fake_image_np = np.repeat(gray, 3, axis=2)              # shape [H,W,3]
        fake_image_np = (fake_image_np * 255).astype(np.uint8)  # Scale back to [0, 255]
        
        # Save the synthesized image
        target_image_path = os.path.join(target_images_dir, os.path.basename(img_path))
        cv2.imwrite(target_image_path, cv2.cvtColor(fake_image_np, cv2.COLOR_RGB2BGR))

# ------------------------------
# 4. Copy Images and Labels
# ------------------------------
def copy_images_and_labels(source_dir, target_dir):
    # Create target directory if it doesn't exist
    os.makedirs(target_dir, exist_ok=True)
    
    # Copy images
    for fname in os.listdir(source_dir):
        if fname.endswith('.png'):
            shutil.copy(os.path.join(source_dir, fname), os.path.join(target_dir, fname))
    
    # Copy labels
    labels_source_dir = source_dir.replace('images', 'labels')
    labels_target_dir = target_dir.replace('images', 'labels')
    os.makedirs(labels_target_dir, exist_ok=True)
    
    for fname in os.listdir(labels_source_dir):
        if fname.endswith('.txt'):
            label_path = os.path.join(labels_source_dir, fname)
            if os.path.exists(label_path):  # Check if the label file exists
                shutil.copy(label_path, os.path.join(labels_target_dir, fname))

# ------------------------------
# 5. Main Function to Generate Dataset
# ------------------------------
def generate_synthesized_dataset_gan():
    # Define base directories
    base_source_dir = "../data/agents"
    base_target_dir = "../data/synthetic_raw/GANs"
    
    # List of agents
    agents = ['agent_0', 'agent_1', 'agent_2']
    
    # Load the trained GAN model
    model_path = "../runs/generative_models/GANs/models/20epochs/generator_best.pth"
    model = load_trained_gan_model(model_path, device)
    
    # Process each agent
    for agent in agents:
        # Define source and target directories for training data
        source_images_dir = os.path.join(base_source_dir, agent, 'train', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'images')
        
        # Synthesize images for the current agent
        synthesize_images_gan(model, source_images_dir, target_images_dir)
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'train', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'train', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)
    
    # Repeat the process for the validation data
    for agent in agents:
        # Define source and target directories for validation data
        source_images_dir = os.path.join(base_source_dir, agent, 'valid', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'images')
        
        # Synthesize images for the current agent
        synthesize_images_gan(model, source_images_dir, target_images_dir)
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'valid', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'valid', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)

    print("Dataset generation complete!")

# Run the dataset generation
if __name__ == "__main__":
    generate_synthesized_dataset_gan()

In [ ]:
import cv2

# Load the image
test_image_path = "../data/synthetic_raw/GANs/agent_0_synthesized/train/images/train_b_cell_11_4_23_cap1_1011.png"
image = cv2.imread(test_image_path)

# Check if the image was loaded successfully
if image is None:
    print("Error: Unable to load image.")
else:
    # Get the number of channels
    num_channels = image.shape[2] if len(image.shape) == 3 else 1
    print(f"The image has {num_channels} channels.")

    # Check if all channels have the same values for each pixel
    if num_channels > 1:
        # Compare all channels
        is_identical = (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 1] == image[:, :, 2]).all()
        if is_identical:
            print("All channels have the same values for each pixel.")
        else:
            print("The channels do not have the same values for each pixel.")
    else:
        print("The image is grayscale (only one channel).")